# Avaliação do Retriever - FAQ Kairos

Este notebook tem como objetivo avaliar e otimizar os parâmetros
do sistema de recuperação da base de conhecimento do FAQ do Kairos.

Serão avaliados, de forma experimental:

- `chunk_size`
- `chunk_overlap`
- `k`
- `fetch_k`
- `lambda_mult`

A avaliação será realizada utilizando um conjunto de perguntas
representativas das possíveis interações do usuário com o agente FAQ.

O objetivo é identificar uma configuração que apresente melhor
desempenho na recuperação dos trechos relevantes da base de conhecimento.

In [1]:
import os
import time
import pandas as pd

from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer

from langchain_core.embeddings import Embeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = "../data/FAQ_Kairos.pdf"

C:\Users\gabrielgarcia-ieg\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [2]:
class E5Embeddings(Embeddings):

    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name)

        self.query_instruction = (
            "Instruct: Given a question, retrieve passages "
            "from the FAQ that answer the question\n"
            "Query: "
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            self.query_instruction + text,
            normalize_embeddings=True
        ).tolist()
        
embeddings = E5Embeddings(
    "intfloat/multilingual-e5-large-instruct"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
evaluation_dataset = [
    {
        "question": "O que é o Kairos?",
        "expected_keywords": ["plataforma", "gôndolas", "supermercados"]
    },
    {
        "question": "Qual é o principal objetivo do Kairos?",
        "expected_keywords": ["rupturas", "reposição", "decisões"]
    },
    {
        "question": "O que é uma ruptura de estoque?",
        "expected_keywords": ["produto", "disponível", "gôndola"]
    },
    {
        "question": "Como o Kairos ajuda a reduzir rupturas?",
        "expected_keywords": ["reposição", "alertas", "repositores"]
    },
    {
        "question": "O Kairos utiliza inteligência artificial?",
        "expected_keywords": ["inteligência", "artificial", "recomendações"]
    },

    # Perfis de usuário
    {
        "question": "Quais são os perfis de usuário do Kairos?",
        "expected_keywords": ["repositor", "vendedor", "cliente", "gerente"]
    },
    {
        "question": "O que o repositor faz no Kairos?",
        "expected_keywords": ["alertas", "reposição", "escaneamento"]
    },
    {
        "question": "O que o gerente faz no Kairos?",
        "expected_keywords": ["gôndolas", "dashboards", "chatbot"]
    },
    {
        "question": "O gerente consegue administrar usuários?",
        "expected_keywords": ["gerente", "vendedores", "repositores"]
    },
    {
        "question": "Todos os usuários possuem as mesmas funcionalidades?",
        "expected_keywords": ["perfil", "funcionalidades"]
    },

    # Monitoramento e reposição
    {
        "question": "Como funciona o monitoramento de reposição?",
        "expected_keywords": ["necessidade", "reposição", "alerta"]
    },
    {
        "question": "Quem recebe os alertas de reposição?",
        "expected_keywords": ["repositor"]
    },
    {
        "question": "Como o repositor confirma uma reposição?",
        "expected_keywords": ["escaneamento", "gôndola"]
    },
    {
        "question": "O Kairos registra quando uma reposição aconteceu?",
        "expected_keywords": ["data", "hora", "histórico"]
    },
    {
        "question": "Existe histórico de reposições?",
        "expected_keywords": ["histórico", "reposição"]
    },

    # Cliente e vendas
    {
        "question": "O Kairos registra vendas?",
        "expected_keywords": ["vendas", "histórico"]
    },
    {
        "question": "O cadastro do e-mail do cliente é obrigatório?",
        "expected_keywords": ["não", "opcional"]
    },
    {
        "question": "O cliente pode visualizar seus produtos favoritos?",
        "expected_keywords": ["produtos", "favoritos"]
    },
    {
        "question": "O cliente pode receber descontos personalizados?",
        "expected_keywords": ["descontos", "histórico", "compras"]
    },
    {
        "question": "O histórico de compras influencia a experiência do cliente?",
        "expected_keywords": ["histórico", "compras", "personalizadas"]
    },

    # Gestão e chatbot
    {
        "question": "O Kairos possui dashboards?",
        "expected_keywords": ["dashboards", "gerentes"]
    },
    {
        "question": "Quem utiliza os dashboards?",
        "expected_keywords": ["gerentes"]
    },
    {
        "question": "Quem utiliza o chatbot especialista?",
        "expected_keywords": ["gerente"]
    },
    {
        "question": "Para que serve o chatbot?",
        "expected_keywords": ["recomendações", "estoque"]
    },
    {
        "question": "O chatbot pode recomendar investimentos em estoque?",
        "expected_keywords": ["recomendações", "investimento", "estoque"]
    },

    # LGPD e privacidade
    {
        "question": "O que é a LGPD?",
        "expected_keywords": ["Lei", "proteção", "dados pessoais"]
    },
    {
        "question": "O e-mail do cliente é um dado pessoal?",
        "expected_keywords": ["e-mail", "dado pessoal"]
    },
    {
        "question": "Quais são os direitos do cliente sobre seus dados?",
        "expected_keywords": ["acesso", "correção", "eliminação"]
    },
    {
        "question": "O cliente pode pedir a exclusão dos seus dados?",
        "expected_keywords": ["eliminação", "dados", "LGPD"]
    },
    {
        "question": "Por quanto tempo meus dados ficam armazenados?",
        "expected_keywords": ["não", "definido", "documentação"]
    }
]

In [4]:
def evaluate_retriever(
    db,
    evaluation_dataset,
    k=6,
    fetch_k=15,
    lambda_mult=0.7
):
    """
    Avalia o desempenho do retriever utilizando MMR.

    Retorna:
        - recall: proporção de perguntas em que os chunks
          recuperados contêm todas as palavras-chave esperadas.
        - hits: quantidade de perguntas recuperadas corretamente.
        - total: quantidade total de perguntas avaliadas.
    """

    hits = 0

    for item in evaluation_dataset:

        question = item["question"]
        expected_keywords = item["expected_keywords"]

        results = db.max_marginal_relevance_search(
            question,
            k=k,
            fetch_k=fetch_k,
            lambda_mult=lambda_mult
        )

        retrieved_text = " ".join(
            result.page_content.lower()
            for result in results
        )

        keywords_found = all(
            keyword.lower() in retrieved_text
            for keyword in expected_keywords
        )

        if keywords_found:
            hits += 1

    total = len(evaluation_dataset)
    recall = hits / total

    return {
        "recall": recall,
        "hits": hits,
        "total": total
    }

In [5]:
def create_faq_database(
    chunk_size,
    chunk_overlap
):
    """
    Cria um banco vetorial do FAQ utilizando
    os parâmetros de chunking informados.
    """

    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_documents(docs)

    db = FAISS.from_documents(
        chunks,
        embeddings
    )

    return db, len(chunks)

In [6]:
baseline_db, baseline_num_chunks = create_faq_database(
    chunk_size=700,
    chunk_overlap=150
)

baseline_evaluation = evaluate_retriever(
    db=baseline_db,
    evaluation_dataset=evaluation_dataset,
    k=6,
    fetch_k=15,
    lambda_mult=0.7
)

print(f"Chunks: {baseline_num_chunks}")
print(f"Acertos: {baseline_evaluation['hits']}")
print(f"Recall@6: {baseline_evaluation['recall']:.2%}")

Chunks: 43
Acertos: 29
Recall@6: 96.67%


## Experimento 1 — Otimização do Chunking

Nesta etapa são avaliadas diferentes combinações de `chunk_size`
e `chunk_overlap`, mantendo os parâmetros do MMR constantes:

- `k = 6`
- `fetch_k = 15`
- `lambda_mult = 0.7`

O objetivo é identificar uma configuração de chunking que maximize
a recuperação de informações relevantes, considerando também a
quantidade de chunks gerados.

In [7]:
chunk_configs = [
    (400, 50),
    (400, 100),
    (400, 150),
    (600, 50),
    (600, 100),
    (600, 150),
    (800, 50),
    (800, 100),
    (800, 150),
]

In [8]:
experiment_results = []

for chunk_size, chunk_overlap in chunk_configs:

    print(
        f"Testando chunk_size={chunk_size}, "
        f"chunk_overlap={chunk_overlap}"
    )

    start_time = time.perf_counter()

    db, num_chunks = create_faq_database(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    embedding_time = time.perf_counter() - start_time

    evaluation = evaluate_retriever(
        db=db,
        evaluation_dataset=evaluation_dataset,
        k=6,
        fetch_k=15,
        lambda_mult=0.7
    )

    experiment_results.append({
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "num_chunks": num_chunks,
        "recall": evaluation["recall"],
        "hits": evaluation["hits"],
        "embedding_time": embedding_time
    })

Testando chunk_size=400, chunk_overlap=50
Testando chunk_size=400, chunk_overlap=100
Testando chunk_size=400, chunk_overlap=150
Testando chunk_size=600, chunk_overlap=50
Testando chunk_size=600, chunk_overlap=100
Testando chunk_size=600, chunk_overlap=150
Testando chunk_size=800, chunk_overlap=50
Testando chunk_size=800, chunk_overlap=100
Testando chunk_size=800, chunk_overlap=150


In [9]:
import pandas as pd

results_df = pd.DataFrame(experiment_results)

results_df.sort_values(
    by=["recall", "num_chunks"],
    ascending=[False, True]
)

,chunk_size,chunk_overlap,num_chunks,recall,hits,embedding_time
3,600,50,43,0.966667,29,110.457125
4,600,100,46,0.966667,29,141.147352
5,600,150,51,0.966667,29,157.386806
2,400,150,86,0.966667,29,382.594075
6,800,50,33,0.933333,28,135.486404
7,800,100,33,0.933333,28,135.003623
8,800,150,34,0.933333,28,155.038942
0,400,50,66,0.900000,27,145.765637
1,400,100,72,0.900000,27,174.550574


### Análise do Experimento

A configuração `chunk_size=600` apresentou o melhor desempenho geral.

As configurações 600/50, 600/100, 600/150 e 400/150
obtiveram Recall@6 de 96,67%. Entretanto, a configuração
600/50 gerou apenas 43 chunks, enquanto as demais geraram
46, 51 e 86 chunks, respectivamente.

Além disso, 600/50 apresentou o menor tempo de processamento
entre as configurações que atingiram o maior Recall.

Dessa forma, `chunk_size=600` e `chunk_overlap=50` foram
selecionados como configuração de chunking para os próximos
experimentos.

## Experimento 2.1 — Otimização do parâmetro k

Nesta etapa, o parâmetro `k` do MMR será avaliado enquanto os
demais parâmetros permanecem fixos.

Configuração utilizada:

- `chunk_size = 600`
- `chunk_overlap = 50`
- `fetch_k = 15`
- `lambda_mult = 0.7`

Serão avaliados os valores de `k`:

`2, 3, 4, 5, 6 e 8`.

O objetivo é identificar a menor quantidade de documentos
recuperados que mantém um bom desempenho de Recall@k.

In [10]:
k_values = [2, 3, 4, 5, 6, 8]

mmr_test_db, mmr_test_num_chunks = create_faq_database(
    chunk_size=600,
    chunk_overlap=50
)

print(f"Quantidade de chunks: {mmr_test_num_chunks}")

Quantidade de chunks: 43


In [11]:
k_results = []

for k in k_values:

    evaluation = evaluate_retriever(
        db=mmr_test_db,
        evaluation_dataset=evaluation_dataset,
        k=k,
        fetch_k=15,
        lambda_mult=0.7
    )

    k_results.append({
        "k": k,
        "recall": evaluation["recall"],
        "hits": evaluation["hits"]
    })

In [12]:
k_results_df = pd.DataFrame(k_results)

k_results_df.sort_values(
    by="recall",
    ascending=False
)

,k,recall,hits
3,5,0.966667,29
2,4,0.966667,29
4,6,0.966667,29
5,8,0.966667,29
1,3,0.900000,27
0,2,0.866667,26


### Análise do Experimento

O aumento de `k` apresentou melhora significativa entre os valores
2 e 4. A partir de `k=4`, o Recall permaneceu em 96,67%, não
havendo ganho adicional com `k=5`, `k=6` ou `k=8`.

Dessa forma, `k=4` foi selecionado por apresentar o maior Recall
obtido utilizando a menor quantidade de documentos recuperados.

Configuração selecionada:

- `k = 4`

## Experimento 2.2 — Otimização do parâmetro fetch_k

Nesta etapa, o parâmetro `fetch_k` será avaliado enquanto os
demais parâmetros permanecem fixos.

Configuração utilizada:

- `chunk_size = 600`
- `chunk_overlap = 50`
- `k = 4`
- `lambda_mult = 0.7`

Serão avaliados os valores de `fetch_k`:

`4, 6, 8, 10, 15 e 20`.

O objetivo é identificar a quantidade de documentos candidatos
necessária para que o MMR selecione um conjunto relevante e
diversificado de resultados.

In [13]:
fetch_k_values = [4, 6, 8, 10, 15, 20]

fetch_k_results = []

for fetch_k in fetch_k_values:

    evaluation = evaluate_retriever(
        db=mmr_test_db,
        evaluation_dataset=evaluation_dataset,
        k=4,
        fetch_k=fetch_k,
        lambda_mult=0.7
    )

    fetch_k_results.append({
        "fetch_k": fetch_k,
        "recall": evaluation["recall"],
        "hits": evaluation["hits"]
    })

In [14]:
fetch_k_results_df = pd.DataFrame(fetch_k_results)

fetch_k_results_df.sort_values(
    by="recall",
    ascending=False
)

,fetch_k,recall,hits
4,15,0.966667,29
3,10,0.966667,29
5,20,0.966667,29
1,6,0.933333,28
2,8,0.933333,28
0,4,0.900000,27


### Análise do Experimento

O aumento de `fetch_k` apresentou melhora no Recall até o valor
10. A partir de `fetch_k=10`, o Recall permaneceu em 96,67%,
não havendo ganho adicional com `fetch_k=15` ou `fetch_k=20`.

Valores inferiores a 10 apresentaram redução no desempenho.

Dessa forma, `fetch_k=10` foi selecionado por atingir o maior
Recall utilizando a menor quantidade de documentos candidatos
entre as configurações avaliadas.

## Experimento 2.3 — Otimização do parâmetro lambda_mult

Nesta etapa, o parâmetro `lambda_mult` do MMR será avaliado
enquanto os demais parâmetros permanecem fixos.

Configuração utilizada:

- `chunk_size = 600`
- `chunk_overlap = 50`
- `k = 4`
- `fetch_k = 10`

Serão avaliados os valores de `lambda_mult`:

`0.0, 0.25, 0.5, 0.7, 0.9 e 1.0`.

O objetivo é identificar o equilíbrio entre relevância dos
documentos recuperados e diversidade dos resultados.

In [15]:
lambda_values = [0.0, 0.25, 0.5, 0.7, 0.9, 1.0]

lambda_results = []

for lambda_mult in lambda_values:

    evaluation = evaluate_retriever(
        db=mmr_test_db,
        evaluation_dataset=evaluation_dataset,
        k=4,
        fetch_k=10,
        lambda_mult=lambda_mult
    )

    lambda_results.append({
        "lambda_mult": lambda_mult,
        "recall": evaluation["recall"],
        "hits": evaluation["hits"]
    })

In [16]:
lambda_results_df = pd.DataFrame(lambda_results)

lambda_results_df.sort_values(
    by="recall",
    ascending=False
)

,lambda_mult,recall,hits
3,0.70,0.966667,29
0,0.00,0.933333,28
1,0.25,0.933333,28
2,0.50,0.900000,27
4,0.90,0.900000,27
5,1.00,0.900000,27


### Análise do Experimento

O parâmetro `lambda_mult` apresentou melhor desempenho no valor
0.7, alcançando 96,67% de Recall.

Valores menores, que priorizam maior diversidade entre os
resultados, apresentaram desempenho inferior. Da mesma forma,
valores próximos de 1.0, que priorizam a similaridade com a
consulta, também apresentaram redução no Recall.

Dessa forma, `lambda_mult=0.7` foi selecionado como o melhor
equilíbrio entre relevância e diversidade para o conjunto de
perguntas utilizado.

In [17]:
final_config = {
    "chunk_size": 600,
    "chunk_overlap": 50,
    "k": 4,
    "fetch_k": 10,
    "lambda_mult": 0.7
}

final_config

{'chunk_size': 600,
 'chunk_overlap': 50,
 'k': 4,
 'fetch_k': 10,
 'lambda_mult': 0.7}

## Configuração otimizada

Após os experimentos de chunking e MMR, foi definida a seguinte
configuração para o retriever:

- `chunk_size = 600`
- `chunk_overlap = 50`
- `k = 4`
- `fetch_k = 10`
- `lambda_mult = 0.7`

Essa configuração apresentou 96,67% de Recall no conjunto de
30 perguntas avaliadas.

In [18]:
final_db, final_num_chunks = create_faq_database(
    chunk_size=600,
    chunk_overlap=50
)

print(f"Quantidade de chunks: {final_num_chunks}")

Quantidade de chunks: 43


In [19]:
final_evaluation = evaluate_retriever(
    db=final_db,
    evaluation_dataset=evaluation_dataset,
    k=4,
    fetch_k=10,
    lambda_mult=0.7
)

print(f"Perguntas avaliadas: {final_evaluation['total']}")
print(f"Acertos: {final_evaluation['hits']}")
print(f"Recall: {final_evaluation['recall']:.2%}")

Perguntas avaliadas: 30
Acertos: 29
Recall: 96.67%


In [20]:
comparison_df = pd.DataFrame([
    {
        "configuracao": "Baseline",
        "chunk_size": 700,
        "chunk_overlap": 150,
        "k": 6,
        "fetch_k": 15,
        "lambda_mult": 0.7,
        "num_chunks": baseline_num_chunks,
        "recall": baseline_evaluation["recall"]
    },
    {
        "configuracao": "Otimizada",
        "chunk_size": 600,
        "chunk_overlap": 50,
        "k": 4,
        "fetch_k": 10,
        "lambda_mult": 0.7,
        "num_chunks": final_num_chunks,
        "recall": final_evaluation["recall"]
    }
])

comparison_df

,configuracao,chunk_size,chunk_overlap,k,fetch_k,lambda_mult,num_chunks,recall
0,Baseline,700,150,6,15,0.7,43,0.966667
1,Otimizada,600,50,4,10,0.7,43,0.966667


## Conclusão

Os experimentos permitiram identificar uma configuração mais
enxuta para o retriever sem perda de desempenho no conjunto de
avaliação.

A configuração baseline apresentou Recall de 96,67%, com 29 das
30 perguntas recuperadas corretamente.

Após os experimentos de chunking e MMR, foi selecionada a seguinte
configuração:

- `chunk_size = 600`
- `chunk_overlap = 50`
- `k = 4`
- `fetch_k = 10`
- `lambda_mult = 0.7`

A configuração otimizada também apresentou Recall de 96,67%
(29/30), mantendo o mesmo desempenho da baseline.

Entretanto, a configuração otimizada reduz `k` de 6 para 4 e
`fetch_k` de 15 para 10. Dessa forma, o retriever recupera e
encaminha uma quantidade menor de documentos ao agente sem perda
de desempenho no conjunto de avaliação.

Portanto, os experimentos demonstram que a configuração inicial
podia ser simplificada sem comprometer o resultado obtido nas
30 perguntas avaliadas.